# **PRUEBA TÉCNICA – ANALISTA DE DATOS OPERATIVO**
Critertec Educación / Soy Digital
Momento 2: Pipeline de integración PWA + Suite Administrativa

In [3]:
import pandas as pd
import numpy as np

Realizamos la carga de los archivos, para este ejercicio, utilizo el siguiente código para cargar los datos en Google Colab

In [4]:
from google.colab import files

uploaded = files.upload()

Saving suite_ferias(in).csv to suite_ferias(in).csv
Saving pwa_usuarios(in).csv to pwa_usuarios(in).csv
Saving suite_sesiones(in).csv to suite_sesiones(in).csv


In [5]:
ferias   = pd.read_csv('suite_ferias(in).csv')
usuarios = pd.read_csv('pwa_usuarios(in).csv')
sesiones = pd.read_csv('suite_sesiones(in).csv')

# Paso 1 - Extracción
Cargamos las tres fuentes. En producción estos paths serían reemplazados por conexiones a la base de datos o a un data lake (S3, GCS, etc.).

In [6]:
print(f"[EXTRACT] Usuarios cargados:  {len(usuarios)}")
print(f"[EXTRACT] Ferias cargadas:    {len(ferias)}")
print(f"[EXTRACT] Sesiones cargadas:  {len(sesiones)}")

[EXTRACT] Usuarios cargados:  30
[EXTRACT] Ferias cargadas:    10
[EXTRACT] Sesiones cargadas:  35


# Paso 2 - Validaciones de calidad
Antes de transformar revisamos integridad de la llave de cruce y campos críticos. Los resultados de estas validaciones alimentan los flags de salida.

Normalizamos "fair_code" mediante espacios y mayúsculas para evitar falsos no-match

In [7]:
usuarios['fair_code_clean'] = usuarios['fair_code'].fillna('').str.strip().str.upper()
ferias['fair_code_clean']   = ferias['fair_code'].fillna('').str.strip().str.upper()
sesiones['fair_code_clean'] = sesiones['fair_code'].fillna('').str.strip().str.upper()

In [8]:
valid_fair_codes = set(ferias['fair_code_clean'])

En el siguiente paso, se identifican los registros de PWA cuyos fair_code no tienen correspondencia en la Suite. Se decide conservar estos registros, es decir, no se eliminan del conjunto de datos. Para ello, se asigna el indicador flag_no_fair_match = 'SI' y los atributos asociados a la feria se mantienen con valores NULL.
Esta decisión se fundamenta en que el usuario existe y aporta valor analítico, aun cuando la feria asociada no esté registrada en la Suite. Excluir estos registros generaría un sesgo de subcobertura, al reducir artificialmente la representación de usuarios válidos dentro del análisis.


In [9]:
def calcular_flag_no_match(fair_code_clean):
    if fair_code_clean == '':
        return 'NO'   # registro autónomo sin feria, comportamiento esperado
    return 'SI' if fair_code_clean not in valid_fair_codes else 'NO'

usuarios['flag_no_fair_match'] = usuarios['fair_code_clean'].apply(calcular_flag_no_match)
huerfanos = usuarios[usuarios['flag_no_fair_match'] == 'SI']
print(f"\n[QA] Usuarios con fair_code sin match en Suite: {len(huerfanos)}")
print(huerfanos[['user_id', 'full_name', 'fair_code']].to_string(index=False))



[QA] Usuarios con fair_code sin match en Suite: 2
user_id            full_name     fair_code
   U009   Elena Jiménez Cruz FAIR-2026-999
   U026 Ramón Sánchez Torres FAIR-2026-999


 Detectar perfiles incompletos en campos críticos: municipio, género, education_level.

**Decisión:** se mantienen en el dataset con flag_incomplete_profile = 'SI'. No se imputan porque son datos autodeclarados; la imputación falsa introduciría ruido en los reportes de segmentación para el BID.

In [10]:
def calcular_flag_incompleto(row):
    campos_criticos = [row.get('municipality'), row.get('gender'), row.get('education_level')]
    return 'SI' if any(pd.isna(v) or str(v).strip() == '' for v in campos_criticos) else 'NO'

usuarios['flag_incomplete_profile'] = usuarios.apply(calcular_flag_incompleto, axis=1)
incompletos = usuarios[usuarios['flag_incomplete_profile'] == 'SI']
print(f"\n[QA] Usuarios con perfil incompleto: {len(incompletos)}")
print(incompletos[['user_id', 'full_name', 'gender', 'municipality', 'education_level']].to_string(index=False))



[QA] Usuarios con perfil incompleto: 3
user_id             full_name   gender             municipality education_level
   U003  Carmen López Sánchez Femenino                La Romana             NaN
   U013 Patricia Torres Núñez Femenino San Francisco de Macorís             NaN
   U023   Sofía León Castillo Femenino                 Barahona             NaN


Verificar consistencia interna de sesiones con ferias


In [11]:
sesiones_huerfanas = set(sesiones['fair_code_clean']) - set(ferias['fair_code_clean'])
print(f"\n[QA] Sesiones con fair_code sin match en ferias: {sesiones_huerfanas or 'ninguna'}")


[QA] Sesiones con fair_code sin match en ferias: ninguna


Al analizar el dataset, se identificó que todos los registros corresponden al tipo INITIAL, por lo que actualmente no existen múltiples evaluaciones para un mismo usuario y no es necesario aplicar ningún criterio de selección. Sin embargo, se documenta que, si en futuras versiones del dataset llegaran a incorporarse registros de tipo REASSESSMENT, se conservaría únicamente la evaluación más reciente de cada usuario, utilizando como referencia la fecha más tardía del campo registration_completed_at. Este criterio se establece con fines de trazabilidad y consistencia metodológica.



In [12]:
print(f"\n[QA] Tipos de diagnóstico: {usuarios['diagnostic_type'].value_counts().to_dict()}")


[QA] Tipos de diagnóstico: {'INITIAL': 30}


# Paso 3 - Transformaciones

Preparamos un lookup de ferias con solo los campos que necesitamos

In [13]:
ferias_lookup = (
    ferias[['fair_code_clean', 'region_nombre', 'status', 'ally_category']]
    .rename(columns={
        'fair_code_clean': 'fair_code_key',
        'region_nombre':   'fair_region',
        'status':          'fair_status',
    })
    .drop_duplicates(subset='fair_code_key')
)

Se utiliza un LEFT JOIN para conservar todos los usuarios del dataset principal, independientemente de que exista o no una coincidencia con la información de ferias. Cuando un usuario no tiene una correspondencia válida entre su fair_code y la Suite, se le asigna flag_no_fair_match = 'SI' y los campos relacionados con la feria (fair_region, fair_status y ally_category) permanecen con valores NULL. De esta manera, se evita la pérdida de usuarios válidos y se identifica claramente cuáles no cuentan con información de feria asociada.

In [14]:
df = usuarios.merge(
    ferias_lookup,
    left_on='fair_code_clean',
    right_on='fair_code_key',
    how='left'
)

Para las variables diagnostic_level_assigned y diagnostic_status, se utilizaron los campos previamente aplanados en el archivo pwa_usuarios.csv. Se identificó que la columna assigned_level_id presenta valores NULL en tres registros cuyo diagnostic_status es ABANDONED. Este comportamiento es consistente con la lógica del proceso, ya que estos usuarios no completaron el diagnóstico y, por tanto, no se les asignó un nivel. En consecuencia, los valores NULL se conservan, dado que aportan información relevante sobre el estado del usuario.

# Paso 4 - Construcción del dataset final

In [15]:
output = pd.DataFrame({
    'user_id':                   df['user_id'],
    'full_name':                 df['full_name'],
    'gender':                    df['gender'],
    'province':                  df['province'],
    'municipality':              df['municipality'],
    'reported_disability':       df['reported_disability'],   # NULL = no reportó discapacidad
    'education_level':           df['education_level'],
    'fair_code':                 df['fair_code_clean'].replace('', np.nan),
    'fair_region':               df['fair_region'],
    'fair_status':               df['fair_status'],
    'ally_category':             df['ally_category'],
    'diagnostic_level_assigned': df['assigned_level_id'],
    'diagnostic_status':         df['diagnostic_status'],
    'levels_completed':          df['levels_completed'],
    'has_certificate':           df['has_certificate'],
    'certificate_type':          df['certificate_type'],
    'flag_no_fair_match':        df['flag_no_fair_match'],
    'flag_incomplete_profile':   df['flag_incomplete_profile'],
})

print(f"\n[TRANSFORM] Dataset final: {output.shape[0]} filas × {output.shape[1]} columnas")



[TRANSFORM] Dataset final: 30 filas × 18 columnas


# Paso 5 - Carga

In [16]:
output.to_csv('dataset_integrado_soy_digital.csv', index=False)
print("[LOAD] Archivo guardado: dataset_integrado_soy_digital.csv")

[LOAD] Archivo guardado: dataset_integrado_soy_digital.csv


In [17]:
output.to_csv('dataset_integrado_soy_digital.csv', index=False)

print("[LOAD] Archivo guardado: dataset_integrado_soy_digital.csv")

files.download('dataset_integrado_soy_digital.csv')

[LOAD] Archivo guardado: dataset_integrado_soy_digital.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusiones del caso

1. fair_code huérfano (FAIR-2026-999): 2 usuarios (U009, U026) registran un
   código de feria que no existe en la Suite Administrativa. Ambos tienen
   at_the_fair=True, lo que indica que el registro ocurrió en el contexto de
   una feria no capturada en el sistema operativo (posiblemente una feria
   piloto o un error de digitación). Decisión: se conservan con
   flag_no_fair_match='SI' y los campos de feria en NULL. Se recomienda
   escalar al equipo de operaciones para verificar si existe la feria
   física y registrarla en la Suite.

2. education_level vacío: 3 usuarias (U003, U013, U023) no tienen nivel
   educativo registrado. Al ser un campo autodeclarado, no se imputa.
   Se marca con flag_incomplete_profile='SI'. Impacta los reportes de
   segmentación por nivel educativo; se recomienda completar en la próxima
   interacción con el beneficiario.

3. reported_disability: 26 de 30 usuarios tienen este campo vacío. Esto es
   el comportamiento esperado (la mayoría no reporta discapacidad); el campo
   NULL equivale a 'no reportada'. No es un problema de calidad sino de
   diseño del formulario.

4. diagnostic_status=ABANDONED: 3 usuarios abandonaron el diagnóstico inicial,
   lo que deja assigned_level_id en NULL. Se conserva el NULL porque no existe
   nivel asignado; forzar un valor introduciría un dato falso.

5. Consistencia sesiones–ferias: todas las sesiones en suite_sesiones.csv
   tienen fair_code y fair_id válidos. No se encontraron inconsistencias
   entre los dos campos desnormalizados (fair_id vs fair_code) en sesiones.

6. Normalización de fair_code: se aplicó strip() y upper() a los códigos de
   feria en las tres fuentes antes del cruce para neutralizar diferencias de
   espaciado o capitalización que podrían generar falsos no-match silenciosos.